# CTM synthetic tracking — analysis notebook (on-the-fly)

The CTM is now coupled to the frame rate: at internal tick `t` it attends to
frame `t // iterations_per_frame` and predicts that single frame's positions.
Total iterations = `n_frames * iterations_per_frame`.

Plots:

1. **Sequence preview** — frame strip with GT object positions overlaid.
2. **Certainty over ticks** — with frame-block boundaries; shaded by per-tick MAE.
3. **Predicted trajectories** — each frame decoded at its most-certain tick within its block.
4. **Within-frame refinement** — for each frame, show the predictions across its `ipf` ticks.
5. **Per-tick position MAE** — single-frame MAE at each tick, with block boundaries.
6. **Position probability heatmaps** — joint (x, y) softmax at each frame's MC tick.
7. **Spatial attention** — where the model looks within the current frame at each tick.
8. **Neuron raster** — post-activations sorted by dominant FFT frequency.
9. **Output-sync trajectory** — PCA/UMAP of the sync vector across ticks.
10. **Synchronisation magnitude** — RMS of action and output sync accumulators.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

#REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
REPO_ROOT = "/users/ghermi/code/continuous-thought-machines"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print('repo root:', os.getcwd())

repo root: /home/ghermi/code/continuous-thought-machines


In [2]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

from models.utils import get_latest_checkpoint
from tasks.tracking.dataset import SyntheticTrackingDataset
from tasks.tracking.utils import prepare_model

CKPT_DIR    = 'logs_jz/tracking/synthetic'   # <- edit me
SAMPLE_INDEX = 0                # which test sample to visualise
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


In [ ]:
ckpt_path = get_latest_checkpoint(CKPT_DIR)
print('loading:', ckpt_path)
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
args = ckpt['args']

test_data = SyntheticTrackingDataset(
    n_samples=args.n_test,
    n_objects=args.n_objects,
    n_frames=args.n_frames,
    img_size=args.img_size,
    n_bins=args.n_bins,
    blob_sigma_px=args.blob_sigma_px,
    velocity_scale=args.velocity_scale,
    in_channels=getattr(args, 'in_channels', 1),
    seed=args.seed + 1,
)

model = prepare_model(args, DEVICE)

# Initialise lazy modules, then load weights.
dummy_frames, _ = test_data[0]
with torch.no_grad():
    model(dummy_frames.unsqueeze(0).to(DEVICE))
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

N   = args.n_objects
T   = args.n_frames
K   = args.n_bins
IPF = args.iterations_per_frame
ITERS = T * IPF
print(f'Loaded iter {ckpt["iteration"]} | N={N} objects, T={T} frames, K={K} bins, '
      f'ipf={IPF}, total ticks={ITERS}')

In [ ]:
# ── Run a single sample with tracking ─────────────────────────────────────
frames, targets = test_data[SAMPLE_INDEX]
in_C = frames.shape[1]
frames_t = frames.unsqueeze(0).to(DEVICE)          # (1, T, C, H, W)

with torch.inference_mode():
    (predictions, certainties, synch_tracks, pre_acts, post_acts,
     attention, frame_indices, kv_spatial_shape) = model(frames_t, track=True)

# Strip the batch dimension (B=1) and move everything to numpy.
predictions = predictions.cpu().numpy()[0]           # (N*2*K, iters)
certainties = certainties.cpu().numpy()[0]           # (2, iters)
synch_out_track    = synch_tracks[0][:, 0]           # (iters, S_out)
synch_action_track = synch_tracks[1][:, 0]           # (iters, S_action)
post_acts = post_acts[:, 0]                          # (iters, D)
# attention: (iters, B=1, heads, query=1, kv_tokens) where kv_tokens = H'*W'
attention = attention[:, 0, :, 0, :]                 # (iters, heads, kv_tokens)
frame_indices = frame_indices                        # (iters,) int

Hp, Wp = kv_spatial_shape
print(f'predictions: {predictions.shape}  attention: {attention.shape}  '
      f'kv_spatial_shape={kv_spatial_shape}')

# Reshape predictions → (N, 2, K, iters)
# Axis meaning: [object, coord(x=0/y=1), bin, tick]
preds_4d = predictions.reshape(N, 2, K, ITERS)

# Decode GT: (T, N, 2) bin indices → normalised coordinates
gt_bins  = targets.numpy()                           # (T, N, 2)
gt_coord = (gt_bins + 0.5) / K                      # (T, N, 2)  in [0,1]

# Per-tick MAE — at tick t, pred is for frame frame_indices[t].
def tick_mae(tick):
    pred_bins  = preds_4d[..., tick].argmax(axis=2)   # (N, 2)
    pred_coord = (pred_bins + 0.5) / K
    f = int(frame_indices[tick])
    return np.abs(pred_coord - gt_coord[f]).mean()

all_maes = np.array([tick_mae(ti) for ti in range(ITERS)])

# Per-frame most-certain tick (within each block of IPF ticks).
cert_blocked = certainties[1].reshape(T, IPF)
rel_mc       = cert_blocked.argmax(axis=1)            # (T,)
mc_per_frame = np.arange(T) * IPF + rel_mc            # (T,)  absolute tick per frame

# Aggregate MAE using each frame's MC tick.
def mae_at_frame_mc():
    coords = np.zeros((N, T, 2))
    for f, ti in enumerate(mc_per_frame):
        pred_bins = preds_4d[..., ti].argmax(axis=2)  # (N, 2)
        coords[:, f] = (pred_bins + 0.5) / K
    err = np.abs(coords - gt_coord.transpose(1, 0, 2))  # (N, T, 2)
    return err.mean(), coords
mc_mae, mc_coords = mae_at_frame_mc()

print(f'Per-frame MC ticks: {mc_per_frame.tolist()}')
print(f'MAE @ frame-MC ticks: {mc_mae:.4f}  |  best per-tick MAE: {all_maes.min():.4f}')

# Crossing detection (x-ordering flips between t=0 and t=T-1)
init_x  = gt_coord[0, :, 0]
final_x = gt_coord[-1, :, 0]
is_crossing = not np.array_equal(np.argsort(init_x), np.argsort(final_x))
print(f'Crossing sample: {is_crossing}')

## 1. Sequence preview

Frame strip with ground-truth object positions overlaid as coloured markers.
Object identity is preserved across frames — the colours are consistent.

In [ ]:
OBJ_COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color'][:N]
H = W = args.img_size

def display_frame(ax, t):
    img = frames[t].numpy()
    if img.shape[0] == 1:
        ax.imshow(img[0], cmap='gray', vmin=0, vmax=1)
    else:
        # Renormalise an ImageNet-normalised RGB tensor for display.
        m = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
        s = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
        rgb = (img * s + m).clip(0, 1).transpose(1, 2, 0)
        ax.imshow(rgb)

cols = T; rows = 1
fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2.5))
axes = np.array(axes).ravel()
for t in range(T):
    ax = axes[t]
    display_frame(ax, t)
    for n in range(N):
        cx = gt_coord[t, n, 0] * W
        cy = gt_coord[t, n, 1] * H
        ax.scatter(cx, cy, c=OBJ_COLORS[n], s=60, marker='x', linewidths=2)
    ax.set_title(f'f{t}', fontsize=8)
    ax.axis('off')

patches = [mpatches.Patch(color=OBJ_COLORS[n], label=f'obj {n}') for n in range(N)]
fig.legend(handles=patches, loc='lower center', ncol=N, fontsize=9)
fig.suptitle(f'Sequence preview   |   crossing={is_crossing}', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## 2. Certainty over ticks

`certainties[1] = 1 − normalised_entropy`. Each frame's `IPF` ticks form a
block (vertical dotted lines). Per-tick MAE is plotted on the right axis;
within each block, the model should converge from low to high certainty as
the position estimate sharpens.

In [ ]:
ticks = np.arange(ITERS)
one_bin = 1.0 / K
is_good = all_maes < one_bin

fig, ax = plt.subplots(figsize=(12, 3))
for ti in ticks:
    ax.axvspan(ti, ti + 1, color='limegreen' if is_good[ti] else 'lightcoral', alpha=0.18)
ax.plot(ticks, certainties[1], 'k-', lw=1.5, label='1 − norm. entropy')
# Frame-block boundaries
for f in range(1, T):
    ax.axvline(f * IPF, color='dimgray', ls=':', lw=1)
# Mark each frame's MC tick
for ti in mc_per_frame:
    ax.axvline(ti + 0.5, color='navy', ls='--', lw=1, alpha=0.8)

ax.set_xlim(0, ITERS)
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel('internal tick (= frame * ipf + within-frame step)')
ax.set_ylabel('certainty')
ax.set_title(f'Certainty over ticks   |   green = MAE < 1 bin ({one_bin:.3f})   |   '
             f'navy = per-frame MC tick')
ax.legend()

ax2 = ax.twinx()
ax2.plot(ticks, all_maes, color='darkorange', lw=1.2, alpha=0.7, label='per-tick MAE')
ax2.set_ylabel('position MAE (normalised)', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')
ax2.legend(loc='upper right')

# Frame index strip on top
ax_top = ax.twiny()
ax_top.set_xlim(ax.get_xlim())
ax_top.set_xticks(np.arange(T) * IPF + IPF / 2)
ax_top.set_xticklabels([f'f{f}' for f in range(T)], fontsize=8)
ax_top.tick_params(axis='x', length=0)

fig.tight_layout()
plt.show()

## 3. Predicted trajectories

Predicted path is built by decoding each frame at its own most-certain tick
within that frame's `IPF`-tick block. Solid = GT, dashed = predicted.

In [ ]:
# mc_coords already computed in cell 5: shape (N, T, 2)
gt_traj = gt_coord.transpose(1, 0, 2)               # (N, T, 2)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect('equal')
ax.invert_yaxis()  # image convention: y increases downward

for n in range(N):
    c = OBJ_COLORS[n]
    ax.plot(gt_traj[n, :, 0], gt_traj[n, :, 1],
            '-o', color=c, lw=2, ms=6, label=f'obj {n} GT')
    ax.plot(mc_coords[n, :, 0], mc_coords[n, :, 1],
            '--x', color=c, lw=1.5, ms=6, alpha=0.75, label=f'obj {n} pred')
    ax.scatter(gt_traj[n, 0, 0], gt_traj[n, 0, 1],
               c=c, s=120, marker='s', zorder=5, edgecolor='k')

ax.set_xlabel('x (normalised)')
ax.set_ylabel('y (normalised, ↓)')
ax.set_title(f'Trajectories @ per-frame MC ticks   |   MAE={mc_mae:.4f}')
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
plt.show()

## 4. Within-frame refinement

For each frame, plot the predicted position at every one of its `IPF` ticks
(colour = within-block tick index). The intra-block trajectory shows how the
model converges its position estimate while attending to that frame.

In [ ]:
fig, axes = plt.subplots(1, T, figsize=(2.6 * T, 3.0))
if T == 1:
    axes = np.array([axes])

cmap_ipf = plt.cm.viridis

for f in range(T):
    ax = axes[f]
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')
    ax.invert_yaxis()

    for n in range(N):
        c = OBJ_COLORS[n]
        # GT for this frame (faint dot)
        ax.scatter(gt_coord[f, n, 0], gt_coord[f, n, 1],
                   c=c, s=120, marker='o', edgecolor='k', alpha=0.8, zorder=5)

        # Within-block predictions
        block_ticks = np.arange(f * IPF, (f + 1) * IPF)
        xs, ys = [], []
        for k, ti in enumerate(block_ticks):
            pred_bins = preds_4d[n, :, :, ti].argmax(axis=1)   # (2,)
            px, py = (pred_bins + 0.5) / K
            xs.append(px); ys.append(py)
            colour = cmap_ipf(k / max(IPF - 1, 1))
            ax.scatter(px, py, c=[colour], s=40, edgecolor=c,
                       linewidths=1.5, zorder=4)
        ax.plot(xs, ys, '-', color=c, lw=1.0, alpha=0.5)

    ax.set_title(f'frame {f}\nticks {f*IPF}–{(f+1)*IPF-1}', fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Within-frame refinement (colour = position in block, ring = object)',
             fontsize=11)
fig.tight_layout()
plt.show()

## 5. Per-tick position MAE

At each tick `t` the model predicts only frame `t // IPF`. The MAE plotted
below is the per-tick error for *that* frame. Frame-block boundaries are
dotted; per-frame MC ticks are dashed navy.

In [ ]:
# Per-object, per-tick MAE
obj_maes = np.zeros((N, ITERS))
for ti in range(ITERS):
    f = int(frame_indices[ti])
    pred_bins  = preds_4d[..., ti].argmax(axis=2)   # (N, 2)
    pred_coord = (pred_bins + 0.5) / K              # (N, 2)
    obj_maes[:, ti] = np.abs(pred_coord - gt_coord[f]).mean(axis=1)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(ticks, all_maes, 'k-', lw=2, label='overall MAE')
for n in range(N):
    ax.plot(ticks, obj_maes[n], '--', color=OBJ_COLORS[n],
            lw=1.3, alpha=0.8, label=f'obj {n} MAE')
ax.axhline(1.0 / K, color='grey', ls=':', lw=1, label='1 bin')
for f in range(1, T):
    ax.axvline(f * IPF, color='dimgray', ls=':', lw=1)
for ti in mc_per_frame:
    ax.axvline(ti + 0.5, color='navy', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('internal tick')
ax.set_ylabel('position MAE (normalised)')
ax.set_title('Per-tick MAE — each tick is for the current frame only')
ax.legend(ncol=N + 2, fontsize=9)
ax.set_xlim(0, ITERS)
fig.tight_layout()
plt.show()

## 6. Position probability heatmaps at per-frame MC ticks

For each (object, frame), the heatmap is the outer product of the x and y
softmax distributions at that frame's most-certain tick. GT is the cyan ✗.

In [ ]:
def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

fig, axes = plt.subplots(N, T, figsize=(1.8 * T, 2.2 * N))
if N == 1:
    axes = axes[np.newaxis, :]

for n in range(N):
    for f in range(T):
        ti = int(mc_per_frame[f])
        ax = axes[n, f]
        x_probs = softmax(preds_4d[n, 0, :, ti])      # (K,)  x-axis
        y_probs = softmax(preds_4d[n, 1, :, ti])      # (K,)  y-axis
        hmap = np.outer(y_probs, x_probs)              # (K, K)
        ax.imshow(hmap, origin='upper', cmap='hot',
                  extent=[0, 1, 1, 0], aspect='equal')
        gx = gt_coord[f, n, 0]
        gy = gt_coord[f, n, 1]
        ax.scatter(gx, gy, c='cyan', s=50, marker='x', linewidths=2, zorder=5)
        ax.set_xticks([]); ax.set_yticks([])
        if f == 0:
            ax.set_ylabel(f'obj {n}', fontsize=8, color=OBJ_COLORS[n])
        if n == 0:
            ax.set_title(f'f{f} (t={ti})', fontsize=8)

fig.suptitle('Joint position probability at per-frame MC ticks   |   cyan ✗ = GT',
             fontsize=11)
fig.tight_layout()
plt.show()

## 7. Spatial attention within the current frame

The CTM attends to one frame per tick. Below: per-frame attention heatmap,
averaged over heads and over the `IPF` ticks within each frame's block,
overlaid on the frame itself. Bright regions = where the model looked
while predicting that frame.

In [ ]:
# attention: (iters, heads, kv_tokens=H'*W')
# Average over heads → (iters, kv_tokens)
attn_iters = attention.mean(axis=1)                  # (iters, H'*W')

# Aggregate per frame block.
attn_per_frame = attn_iters.reshape(T, IPF, Hp * Wp).mean(axis=1)  # (T, H'*W')

fig, axes = plt.subplots(2, T, figsize=(2 * T, 4.2))
if T == 1:
    axes = axes.reshape(2, 1)

for f in range(T):
    # Top row: frame with GT marker.
    display_frame(axes[0, f], f)
    for n in range(N):
        cx = gt_coord[f, n, 0] * W
        cy = gt_coord[f, n, 1] * H
        axes[0, f].scatter(cx, cy, c=OBJ_COLORS[n], s=40, marker='x', linewidths=1.5)
    axes[0, f].set_title(f'f{f}', fontsize=8)
    axes[0, f].axis('off')

    # Bottom row: attention heatmap upsampled to frame resolution, overlaid.
    amap = attn_per_frame[f].reshape(Hp, Wp)
    axes[1, f].imshow(amap, cmap='viridis',
                      extent=[0, W, H, 0], aspect='auto', interpolation='bilinear')
    for n in range(N):
        cx = gt_coord[f, n, 0] * W
        cy = gt_coord[f, n, 1] * H
        axes[1, f].scatter(cx, cy, c=OBJ_COLORS[n], s=40, marker='x', linewidths=1.5)
    axes[1, f].set_title(f'attn f{f}', fontsize=8)
    axes[1, f].axis('off')

fig.suptitle("Spatial attention per frame (avg over heads & within-frame ticks)",
             fontsize=11)
fig.tight_layout()
plt.show()

# Also show the raw (iters × kv_tokens) heatmap so we can see how attention
# evolves across ticks within each frame block.
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(attn_iters.T, aspect='auto', cmap='viridis', origin='upper')
ax.set_xlabel('internal tick')
ax.set_ylabel('spatial token (row-major H′×W′)')
for f in range(1, T):
    ax.axvline(f * IPF - 0.5, color='white', lw=0.8, ls='--', alpha=0.6)
ax_top = ax.twiny()
ax_top.set_xlim(ax.get_xlim())
ax_top.set_xticks(np.arange(T) * IPF + IPF / 2 - 0.5)
ax_top.set_xticklabels([f'f{f}' for f in range(T)], fontsize=8)
ax_top.tick_params(axis='x', length=0)
plt.colorbar(im, ax=ax, shrink=0.8, label='attention weight')
ax.set_title(f'Attention over spatial tokens across ticks (H′×W′ = {Hp}×{Wp})')
fig.tight_layout()
plt.show()

## 8. Neuron raster

Post-activation of a subset of CTM neurons across internal ticks, sorted by
dominant FFT frequency. The key CTM claim: different neurons develop
different temporal response profiles (oscillators, ramps, bursts).

In [ ]:
N_NEURONS = min(64, post_acts.shape[1])

fft_mag  = np.abs(np.fft.rfft(post_acts, axis=0))[1:]   # skip DC; (freqs, D)
peak_bin = fft_mag.argmax(axis=0)                        # (D,)
order    = np.argsort(peak_bin)
chosen   = order[np.linspace(0, len(order) - 1, N_NEURONS).astype(int)]
raster   = post_acts[:, chosen].T                        # (N_NEURONS, iters)

fig, ax = plt.subplots(figsize=(12, 6))
lim = max(np.abs(raster).max(), 1e-6)
im  = ax.imshow(raster, aspect='auto', cmap='RdBu_r',
                vmin=-lim, vmax=lim, interpolation='nearest')
for f in range(1, T):
    ax.axvline(f * IPF - 0.5, color='yellow', lw=0.8, ls='--', alpha=0.7)
ax_top = ax.twiny()
ax_top.set_xlim(ax.get_xlim())
ax_top.set_xticks(np.arange(T) * IPF + IPF / 2 - 0.5)
ax_top.set_xticklabels([f'f{f}' for f in range(T)], fontsize=8)
ax_top.tick_params(axis='x', length=0)
ax.set_xlabel('internal tick')
ax.set_ylabel('neuron (sorted by dominant FFT freq)')
ax.set_title('Per-neuron post-activation raster   |   yellow = frame boundary')
plt.colorbar(im, ax=ax, shrink=0.7)
fig.tight_layout()
plt.show()

## 9. Output-sync trajectory (PCA / UMAP)

The output-synchronisation vector is what drives the position predictions.
Its trajectory through a 2-D projection shows how the latent representation
evolves: early ticks explore, later ticks settle.

In [ ]:
sync = synch_out_track                              # (iters, S_out)

try:
    import umap
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=min(15, max(2, ITERS - 1)),
        min_dist=0.4, metric='cosine', random_state=0,
    )
    emb = reducer.fit_transform(sync)
    method = 'UMAP'
except Exception as e:
    print('UMAP unavailable, falling back to PCA:', e)
    from sklearn.decomposition import PCA
    emb = PCA(n_components=2).fit_transform(sync)
    method = 'PCA'

fig, ax = plt.subplots(figsize=(6, 6))
cmap = plt.cm.viridis
for i in range(len(emb) - 1):
    ax.plot(emb[i:i+2, 0], emb[i:i+2, 1], '-',
            color=cmap(i / max(1, len(emb) - 1)), lw=1.8)
sc = ax.scatter(emb[:, 0], emb[:, 1], c=np.arange(ITERS),
                cmap='viridis', s=40, edgecolor='k', lw=0.5, zorder=3)
# Mark frame-boundary ticks (start of each new frame block).
for f in range(T):
    bi = f * IPF
    ax.scatter(*emb[bi], c='red', edgecolor='black', s=80, marker='s', zorder=5)
ax.scatter(*emb[0],  c='white', edgecolor='black', s=130, zorder=6, label='start')
ax.scatter(*emb[-1], c='black', edgecolor='white', s=100, zorder=6, label='end')
plt.colorbar(sc, ax=ax, label='tick')
ax.legend(fontsize=9)
ax.set_title(f'Output-sync trajectory ({method}, red ▪ = frame onset)')
fig.tight_layout()
plt.show()

## 10. Synchronisation magnitude

RMS of the action-sync and output-sync vectors per tick. The leaky
accumulators build up evidence as more ticks process the same sequence;
the shape reflects the learned time constants.

In [ ]:
sa = np.sqrt((synch_action_track ** 2).mean(axis=-1))   # (iters,)
so = np.sqrt((synch_out_track    ** 2).mean(axis=-1))   # (iters,)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(ticks, sa, label='action sync RMS', color='tab:orange', lw=1.5)
ax.plot(ticks, so, label='output sync RMS', color='tab:blue',   lw=1.5)
for f in range(1, T):
    ax.axvline(f * IPF, color='dimgray', ls=':', lw=1)
ax.set_xlabel('internal tick')
ax.set_ylabel('||sync||_rms')
ax.set_title('Synchronisation accumulator magnitude over ticks')
ax.set_xlim(0, ITERS)
ax.legend()
fig.tight_layout()
plt.show()